# NoteWeaver: Generate, Review, and Export Notes

This English walkthrough connects the real PDF reader to note generation,
editing, approval, verified persistence, and PDF export. Install `.[preview,export]`. Follow [notebook setup](../docs/NOTEBOOK_DEMO.md).

The default mode is **offline**, which uses fixed test content and makes no API call.
For a real answer, change only the `MODE = "offline"` assignment in the first code
cell to `MODE = "openrouter"` or `MODE = "groq"`. Do not edit the `if MODE == ...`
conditions in the provider cell.

- [OpenRouter setup](../docs/OPENROUTER_SETUP.md): `openrouter/free` selects an
  available free model; the underlying model can vary and quotas apply.
- [Groq setup](../docs/GROQ_SETUP.md): uses your Groq account and its quotas.

Both live modes send the selected lecture text and question to the selected
service. OpenRouter may forward them to a model provider. The notebook does not
retry automatically or switch services on failure. Manual export needs no key.

Keep this notebook inside the checkout. Committed outputs are cleared. The
notebook creates a unique folder under ignored `generated/groq-notebook/`.

In [ ]:
import os
from getpass import getpass
from pathlib import Path
import tempfile
from IPython.display import Image, display
from noteweaver import (
    ContractError, FakeNoteProvider, GenerationError, GroqProvider, OpenAICompatibleProvider,
    approve_note, create_project, generate_note, load_project, save_project, update_note,
)
from noteweaver.pdf_reader import PDFReader

MODE = "offline"  # Choose "offline", "openrouter", or "groq" here only.
QUESTION = "Why can a free choice still have an opportunity cost?"
LANGUAGE = "en"  # Use "zh-Hant" for Traditional Chinese output.
ORIGINAL_PAGE = 2

root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "examples/data/sample_lecture_3p.pdf").is_file()
             and (p / "src/noteweaver/workflow.py").is_file()), None)
if root is None:
    raise RuntimeError("Open this notebook inside the NoteWeaver checkout.")
source = root / "examples/data/sample_lecture_3p.pdf"
output_root = root / "generated/groq-notebook"
output_root.mkdir(parents=True, exist_ok=True)
run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=output_root))
reader = PDFReader(preview_dir=run_dir / "previews")
project = create_project(reader.inspect(source), source)
latest_note_id = None

## 1. Select and inspect the original lecture page

The demo uses the real bilingual lecture fixture. Change `source` to your own
text-based PDF when ready. Only the selected page is sent in this example.

In [ ]:
page = reader.read_page(source, ORIGINAL_PAGE)
print(f"Original page {page.original_page}: {page.extraction_status}")
print(page.text)
if page.preview_path is not None:
    display(Image(filename=page.preview_path, width=800))
else:
    print("Preview unavailable; extracted text remains available.")

## 2. Choose the provider

Run the first code cell after changing `MODE`, then this cell, then the generation
cell. Execution numbers show the order cells last ran; an old error below is not
updated just by re-entering a key.

OpenRouter mode uses `OPENROUTER_API_KEY` or a hidden prompt. Groq mode uses
`GROQ_API_KEY` or a hidden prompt. Enter the complete key for the selected service,
not its name, ID, or abbreviated display. Never put a key in a code cell or commit it.
Selecting a provider does not validate the key; the next cell makes the request.
Restarting the kernel clears prompted credentials, but not independently configured
environment variables. Do not share a kernel session containing credentials.


In [ ]:
provider = None
if MODE == "offline":
    provider = FakeNoteProvider()
    print("OFFLINE: fixed test content; no API call will be made.")
elif MODE == "openrouter":
    entered_key = os.environ.get("OPENROUTER_API_KEY", "").strip()
    if not entered_key:
        entered_key = getpass("Enter your OpenRouter API key (hidden): ")
    try:
        if entered_key.strip():
            provider = OpenAICompatibleProvider(
                api_key=entered_key.strip(),
                base_url="https://openrouter.ai/api/v1",
                model="openrouter/free",
                timeout=120,
            )
            print("OpenRouter selected (openrouter/free; model may vary).")
            print("The next cell sends the selected lecture text and question.")
        else:
            print("No key entered. Generation is disabled.")
    finally:
        del entered_key
elif MODE == "groq":
    if os.environ.get("GROQ_API_KEY"):
        provider = GroqProvider()
    else:
        entered_key = getpass("Enter your Groq API key (hidden): ")
        if entered_key.strip():
            provider = GroqProvider(api_key=entered_key)
        else:
            print("No key entered. Generation is disabled.")
        del entered_key
    if provider is not None:
        print("Groq selected. The next cell sends the page text and question.")
else:
    raise ValueError("MODE must be 'offline', 'openrouter', or 'groq'.")

## 3. Generate a draft

Run this cell to generate. In either live mode, every execution makes a new request and
uses quota. No automatic retries or fallback calls are made. A successful call
appends a new draft and its question record; it does not replace previous notes.
On failure, existing project data remains unchanged.

In [ ]:
latest_note_id = None
if provider is None:
    print("Configure a provider in the previous cell first.")
else:
    print("Generating...")
    try:
        updated = generate_note(
            project, original_page=ORIGINAL_PAGE, question=QUESTION,
            provider=provider, language=LANGUAGE, style="example",
        )
    except (ContractError, GenerationError, OSError) as error:
        print(f"Generation failed: {error}")
        print("Existing notes are unchanged. Fix the issue and rerun to retry.")
    else:
        project = updated
        note = project.notes[-1]
        latest_note_id = note.note_id
        print(f"Draft created: {note.title}")
        for section in note.sections:
            print(f"\n[{section.origin} / {section.kind}]\n{section.content}")

## 4. Optionally edit the note

Set `REVISED_CONTENT` to your own explanation, then run this cell. Replacing the
body labels it as user-authored and clears its previous source claims. Editing
resets approval to draft while keeping the note ID and original-page anchor.

In [ ]:
REVISED_CONTENT = None  # Replace with a string to edit the entire note body.
if latest_note_id is None:
    print("No new draft is available.")
elif REVISED_CONTENT is not None:
    project = update_note(project, latest_note_id, content=REVISED_CONTENT)
    print("Edited; the note is a draft and needs review.")
else:
    print("No edit requested.")
if latest_note_id is not None:
    note = next(n for n in project.notes if n.note_id == latest_note_id)
    print(f"\nReview: {note.title} | Status: {note.status}")
    for section in note.sections:
        print(f"\n[{section.origin}] {section.content}")

## 5. Confirm only after reviewing

Change `CONFIRM_REVIEW` to `True` only after reading the displayed note.
Approval records your decision, not a factual correctness guarantee. The
notebook defaults to leaving generated content as a draft, including Run All.
After confirming, run the save, export, and preview cells below. Export never
approves a note on your behalf. A default Run All saves a draft and skips PDF export.


In [ ]:
CONFIRM_REVIEW = False
if CONFIRM_REVIEW and latest_note_id is not None:
    project = approve_note(project, latest_note_id)
    print("Note approved.")
else:
    print("No approval recorded; current review state is preserved.")

## 6. Save and reopen your project

Both drafts and approved notes can be saved. Every execution saves to a new file
in this run folder; no existing project is overwritten. The source PDF must
remain available for verified reopening. This saves JSON, not a study PDF.

In [ ]:
from uuid import uuid4
if project.notes:
    project_file = save_project(project, run_dir / f"study-{uuid4().hex[:8]}.json")
    reopened = load_project(project_file)
    assert reopened == project
    print("Saved and verified:", project_file.relative_to(root))
    print("Note states:", [n.status for n in reopened.notes])
    print("Questions, note IDs, content, and original-page anchors preserved.")
else:
    print("No notes to save yet. Complete generation first.")

## 7. Export approved notes to a study PDF

This exports **all currently approved notes** in the project, in their stored order.
Drafts are excluded. It never approves notes or calls an AI provider. For the
bundled short manual example, the order is original pages 1 and 2, the note page,
then original page 3; longer notes can span more pages.

Each execution uses a new `export-*` folder under this run. It saves the current
project snapshot and a new `study.pdf`, checks the page maps, and verifies the
original source. Earlier exports are never overwritten. After editing a note,
review it again before exporting; an older approval does not apply to the edit.



In [ ]:
from dataclasses import replace
from noteweaver import PyMuPDFPDFExporter, list_notes, validate_export

# Clear the latest result before each attempt so a failure cannot reuse old links.
export_result = None
export_project = None
export_project_file = None
export_dir = None
approved_notes = list_notes(project, status="approved")
print(f"Approved notes: {len(approved_notes)} | Drafts excluded: {len(project.notes) - len(approved_notes)}")
if not approved_notes:
    print("No approved notes to export. Review and confirm a note, then rerun this cell.")
else:
    try:
        exporter = PyMuPDFPDFExporter()
        export_dir = Path(tempfile.mkdtemp(prefix="export-", dir=run_dir))
        # Save the CURRENT project, not an older reopened variable.
        snapshot_file = save_project(project, export_dir / "study.json")
        snapshot = load_project(snapshot_file)
        assert Path(snapshot.source_path).resolve() == Path(project.source_path).resolve()
        assert replace(snapshot, source_path=project.source_path) == project
        result = exporter.export(
            Path(snapshot.source_path), snapshot, export_dir / "study.pdf"
        )
        validate_export(snapshot, result)
    except (ImportError, ContractError, OSError, ValueError) as error:
        print(f"Export failed: {error}")
        print("Existing notes and earlier exports are unchanged. Fix the issue and rerun.")
    else:
        export_result = result
        export_project = project
        export_project_file = snapshot_file
        print("Study PDF:", Path(result.output_path).relative_to(root))
        print("Project snapshot:", snapshot_file.relative_to(root))
        print("Total output pages:", result.output_page_count)
        print("Original page -> output page:",
              [(p.original_page, p.output_page) for p in result.original_to_output_pages])
        print("Note ID -> output pages:",
              [(p.note_id, p.output_pages) for p in result.note_to_output_pages])


## 8. Preview and download the result

The preview below is rendered from the **exported PDF**, not from the source or
a fabricated image. The first supplementary page is selected by its output page
map. Change `PREVIEW_OUTPUT_PAGE` in the next cell to inspect another output page.
Original-page anchors and output-page positions are different after insertion.

The links download the PDF and its editable project snapshot. Keep the original
lecture too: JSON does not embed it. Browser policies can restrict embedded
links; the same files are also available in the printed folder in Jupyter's file
browser. If the project has changed, export again before refreshing this preview.



In [ ]:
from base64 import b64encode
from IPython.display import HTML
from noteweaver.pdf_reader import PreviewError

if export_result is None:
    print("No PDF from the latest export attempt. Complete the export step first.")
elif project != export_project:
    print("The project changed after export. Review any edits and export again for an up-to-date PDF.")
else:
    # Embedded links work independently of the notebook server root or kernel directory.
    pdf_payload = b64encode(Path(export_result.output_path).read_bytes()).decode("ascii")
    json_payload = b64encode(export_project_file.read_bytes()).decode("ascii")
    display(HTML(
        '<p><a download="noteweaver-study.pdf" href="data:application/pdf;base64,'
        + pdf_payload + '">Download study PDF</a> | '
        '<a download="noteweaver-project.json" href="data:application/json;base64,'
        + json_payload + '">Download project JSON</a></p>'
    ))
    print("These downloads are a snapshot of the last successful export.")
    print("If your browser blocks embedded downloads, open this folder in Jupyter's file browser:")
    print(export_dir.relative_to(root))
    # Change this to another OUTPUT page number to inspect more pages.
    PREVIEW_OUTPUT_PAGE = export_result.note_to_output_pages[0].output_pages[0]
    try:
        exported_preview = PDFReader(preview_dir=export_dir / "previews").preview_page(
            Path(export_result.output_path), PREVIEW_OUTPUT_PAGE, scale=1.3
        )
    except (PreviewError, ContractError, OSError, ValueError) as error:
        print(f"Preview unavailable: {error}")
        print("The exported PDF is still available through the download link or file browser.")
    else:
        print(f"Exported PDF preview: output page {PREVIEW_OUTPUT_PAGE}")
        display(Image(filename=exported_preview.preview_path, width=800))


## Next steps

The same public `generate_note()` operation is usable from any Python script.
Use `context_pages=(1, 2, 3)` if you explicitly want to send neighboring pages as
well; these must all contain successfully extracted text. An empty/scanned page
needs manual notes or another extraction solution; this demo does not perform OCR.

AI generation requires a real provider; offline output is only an integration
test double. After explicit review, this notebook exports approved notes and shows PDF/JSON downloads.
Each export uses a new folder. Clear outputs before committing; downloads contain note content.